# Algorithm Logical Estimates

Notebook for generating logical resource estimates for the quantum algorithms implemented in the QTT.

The purpose of this notebook is to give logical estimates as logical qubit and $T$ count estimates. Note that the conversion of different non-Clifford counts to $T$ counts is done by qsharp, including with respect to Toffoli (CCZ) gates and arbitrary rotations.

In [10]:
import json
from pathlib import Path

import pandas as pd
from qsharp.estimator import EstimatorParams

from quantumthreattracker.algorithms.algorithm_lister import AlgorithmLister
from quantumthreattracker.algorithms.quantum_algorithm import CryptParams
from quantumthreattracker.optimizer import AlgorithmOptimizer

In [ ]:
protocols = [
    {"algorithm": "RSA", "keySize": 1024},
    {"algorithm": "RSA", "keySize": 2048},
    {"algorithm": "RSA", "keySize": 4096},
    {"algorithm": "RSA", "keySize": 8192},
    {"algorithm": "DH-SP", "keySize": 1024},
    {"algorithm": "DH-SP", "keySize": 2048},
    {"algorithm": "DH-SP", "keySize": 4096},
    {"algorithm": "DH-SP", "keySize": 8192},
    {"algorithm": "DH-SCH", "keySize": 1024},
    {"algorithm": "DH-SCH", "keySize": 2048},
    {"algorithm": "DH-SCH", "keySize": 4096},
    {"algorithm": "DH-SCH", "keySize": 8192},
    {"algorithm": "ECDH", "keySize": 256},
    {"algorithm": "ECDH", "keySize": 384},
    {"algorithm": "ECDH", "keySize": 512},
    ]

# Data Generation

In [12]:
estimator_params = EstimatorParams()

minimize_metric = "physicalQubits"

results = []

for protocol in protocols:
    crypt_algorithm = protocol["algorithm"]
    key_size = protocol["keySize"]
    crypt_params = CryptParams(crypt_algorithm, key_size)
    for algorithm in AlgorithmLister().list_algorithms(crypt_params):
        alg_params = AlgorithmOptimizer.find_min_estimate(algorithm, estimator_params, minimize_metric)[0]
        estimate = algorithm.estimate_resources_azure(estimator_params=estimator_params, alg_params=alg_params)
        name = algorithm.__class__.__name__
        logical_qubits = estimate["logicalCounts"]["numQubits"]
        t_states = estimate["physicalCounts"]["breakdown"]["numTstates"]
        results.append({
            "protocol": crypt_algorithm,
            "key_size": key_size,
            "algorithm": name,
            "logical_qubits": logical_qubits,
            "t_states": t_states,
        })

df_protocol_algorithms = pd.DataFrame(results)

df_protocol_algorithms.head()

,protocol,key_size,algorithm,logical_qubits,t_states
0,RSA,1024,GidneyBasic,742,4400000000
1,RSA,1024,GidneyEkera,3072,1609350352
2,RSA,2048,GidneyBasic,1399,26000000000
3,RSA,2048,GidneyEkera,6144,10733223936
4,RSA,4096,GidneyBasic,2692,160000000000


# Exporting the Data

In [13]:
# Find repo root by looking for pyproject.toml
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())

data_dir = repo_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

output_path = data_dir / "algorithm_logical_estimates.json"
with output_path.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Saved results to: {output_path}")

Saved results to: c:\Users\walde\Documents\GitHub\QuantumThreatTracker\data\algorithm_logical_estimates.json
